In [1]:
from pathlib import Path

from caf.base import DVector, ZoningSystem
import pandas as pd
from numpy.polynomial.polynomial import polyfit
import matplotlib.pyplot as plt

In [2]:
# main working directory
working_dir = Path(r"I:\NorMITs NorCOM\Validation\post-adjustment\v38\2023")

In [3]:
# define path to 2+ car factors by LSOA from the DVLA Comparison.ipynb
car_factor_dir = working_dir / "dvla" / "2+factor_data"
# define path to total dvla vehicles from the DVLA Comparison.ipynb
dvla_data_dir = working_dir / "dvla" / "dvla_data"
# define path to norcom application outputs from the validate_norcom.py
model_output_dir = working_dir / "data"

In [4]:
# define plotting function
def plot_scatter(df, x_col, y_col, x_label, y_label, title, upper_lim, output_dir, file):
    plt.scatter(
        x=df[x_col], 
        y=df[y_col], 
        s=0.5, 
        c="teal"
    )
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.title(title)
    plt.xlim([0, upper_lim])
    plt.ylim([0, upper_lim])
    
    i, s = polyfit(df[x_col], df[y_col], 1)
    r2 = df[x_col].corr(df[y_col]) ** 2
    plt.plot(df[x_col], s * df[x_col] + i, ls='--', linewidth=1, color='black', label=u'y={:.2f}x+{:.2f}, r\u00B2={:.3f}'.format(s, i, r2))
    plt.legend(loc='best')

    output_dir.mkdir(exist_ok=True, parents=True)
    plt.savefig(output_dir / file, dpi=220, bbox_inches='tight')
    plt.close()

In [5]:
# define output folder to save analysis in
output_dir = working_dir / "dvla" / "total_comparison"
output_dir.mkdir(exist_ok=True)

In [6]:
# create combined lists of all LSOAs
all_expected = []
all_dvla = []

In [7]:
for factor_file in car_factor_dir.glob("*.hdf"):
    gor = factor_file.with_suffix("").name.split("_")[-1]
    factors = DVector.load(factor_file)
    model_data = DVector.load(model_output_dir / f"applied_{gor}.hdf")

    expected_cars = factors * model_data
    expected_cars = expected_cars.add_segments(["total"]).aggregate(["total"])

    dvla_cars = DVector.load(dvla_data_dir / f"2023_dvla_{gor}.hdf")

    expected = expected_cars.data.T
    dvla = dvla_cars.data.T

    expected.columns = ['calculated']
    dvla.columns = ['dvla']

    expected.index.name = "LSOA2021"
    dvla.index.name = "LSOA2021"

    expected.to_csv(output_dir / f"expected_{gor}.csv")
    dvla.to_csv(output_dir / f"dvla_{gor}.csv")

    combined = pd.concat([dvla, expected]).fillna(0).groupby("LSOA2021").sum()
    upper_lim = max(combined["dvla"].max(), combined["calculated"].max())

    plot_scatter(
        df=combined, 
        x_col="dvla", 
        y_col="calculated", 
        x_label="DVLA Registered Cars", 
        y_label="NorCOM Calculated Cars", 
        title="2023 LSOA Car Comparison", 
        upper_lim=upper_lim, 
        output_dir=output_dir / "figs", 
        file=f"car_comparison_{gor}.png"
    )
    
    all_expected.append(expected)
    all_dvla.append(dvla)

all_expected = pd.concat(all_expected)
all_expected.to_csv(output_dir / f"expected.csv")
all_dvla = pd.concat(all_dvla)
all_dvla.to_csv(output_dir / f"dvla.csv")

combined = pd.concat([all_dvla, all_expected]).fillna(0).groupby("LSOA2021").sum()
upper_lim = max(combined["dvla"].max(), combined["calculated"].max())

plot_scatter(
    df=combined, 
    x_col="dvla", 
    y_col="calculated", 
    x_label="DVLA Registered Cars", 
    y_label="NorCOM Calculated Cars", 
    title="2023 LSOA Car Comparison", 
    upper_lim=upper_lim, 
    output_dir=output_dir / "figs", 
    file=f"car_comparison.png"
)